# Baseline vs RAG Evaluation
The objective of this notebook is to compare answers generated by a standalone language model against answers generated by the Retrieval-Augmented Generation (RAG) system.

The comparison evaluates whether retrieval improves answer accuracy, specificity and grounding when answering questions from university documents.

This experiment directly addresses the research question:

Can Retrieval-Augmented Generation improve the accuracy and relevance of answers generated from university documents compared to a standalone language model?

In [2]:
!pip install -q pypdf
!pip install -q sentence-transformers
!pip install -q langchain-text-splitters
!pip install -q transformers
!pip install -q torch
!pip install -q faiss-cpu

In [3]:
# Import libraries used for baseline and RAG evaluation

from pathlib import Path

from pypdf import PdfReader

from langchain_text_splitters import RecursiveCharacterTextSplitter

from sentence_transformers import SentenceTransformer

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM
)

import faiss
import numpy as np
import pandas as pd

In [6]:
# Load all PDF documents from the university knowledge base

pdf_files = list(
    Path(
        "/kaggle/input/datasets/makhosazanemthethwa/cos802-university-documents"
    ).glob("*.pdf")
)

print(f"Documents Loaded: {len(pdf_files)}")

for pdf in pdf_files:
    print(pdf.name)

Documents Loaded: 9
COS802_Project_Description.pdf
COS802_Study_Guide_2026.pdf
Code_of_Conduct_for_Students.pdf
EBIT_Faculty_Rules_2026.pdf
General_Academic_Regulations_2026.pdf
Policy_on_Examinations_and_Related_Matters.pdf
Programme_Yearbook_12254017.pdf
UKZN_Student_Rulebook_2026.pdf
UP_Information_Guide_2026.pdf


In [7]:
# Extract text from every document and preserve source metadata

documents = []

for pdf_file in pdf_files:

    reader = PdfReader(str(pdf_file))

    text = ""

    for page in reader.pages:

        page_text = page.extract_text()

        if page_text:
            text += page_text + "\n"

    documents.append(
        {
            "source": pdf_file.name,
            "text": text
        }
    )

print(f"Documents Processed: {len(documents)}")

Documents Processed: 9


In [8]:
# Split documents into chunks while preserving source information

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = []
sources = []

for doc in documents:

    split_chunks = text_splitter.split_text(
        doc["text"]
    )

    chunks.extend(split_chunks)

    sources.extend(
        [doc["source"]] * len(split_chunks)
    )

print(f"Total Chunks: {len(chunks)}")
print(f"Total Sources: {len(sources)}")

Total Chunks: 1062
Total Sources: 1062


In [9]:
# Load the Sentence Transformer model

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded


In [10]:
# Generate embeddings for all chunks

embeddings = embedding_model.encode(
    chunks,
    show_progress_bar=True
)

embeddings = np.array(
    embeddings
).astype("float32")

print(f"Embeddings Generated: {len(embeddings)}")
print(f"Embedding Dimension: {embeddings.shape[1]}")

Batches:   0%|          | 0/34 [00:00<?, ?it/s]

Embeddings Generated: 1062
Embedding Dimension: 384


In [11]:
# Create FAISS vector index

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(embeddings)

print(f"Vectors Stored: {index.ntotal}")

Vectors Stored: 1062


In [12]:
# Load the FLAN-T5 model used for answer generation

model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

llm = AutoModelForSeq2SeqLM.from_pretrained(model_name)

print("LLM loaded successfully")

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

... (2 outputs left out)

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

LLM loaded successfully


In [13]:
# Retrieve the most relevant document chunks

def retrieve_context(question, top_k=3):

    query_embedding = embedding_model.encode([question])

    query_embedding = np.array(
        query_embedding
    ).astype("float32")

    distances, indices = index.search(
        query_embedding,
        top_k
    )

    retrieved_chunks = [
        chunks[idx]
        for idx in indices[0]
    ]

    return "\n\n".join(retrieved_chunks)

In [17]:
# Generate an answer without retrieval

def generate_baseline_answer(question):

    prompt = f"""
    Answer the following question.

    Question:
    {question}

    Answer:
    """

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    outputs = llm.generate(
        **inputs,
        max_new_tokens=100
    )

    answer = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return answer

In [18]:
# Generate an answer using retrieval

def generate_rag_answer(question):

    context = retrieve_context(
        question,
        top_k=3
    )

    prompt = f"""
    Use only the provided context to answer the question.

    Context:
    {context}

    Question:
    {question}

    Answer:
    """

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    )

    outputs = llm.generate(
        **inputs,
        max_new_tokens=100
    )

    answer = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return answer

In [19]:
question = "What is the prerequisite for COS802?"

baseline_answer = generate_baseline_answer(question)

rag_answer = generate_rag_answer(question)

print("QUESTION:")
print(question)

print("\nBASELINE ANSWER:")
print(baseline_answer)

print("\nRAG ANSWER:")
print(rag_answer)

QUESTION:
What is the prerequisite for COS802?

BASELINE ANSWER:
a fusion reaction

RAG ANSWER:
MIT 801 [Intro to Machine Learning]


## Observations
A direct comparison between a standalone language model and the Retrieval-Augmented Generation (RAG) system was performed using the question:

"What is the prerequisite for COS802?"

The standalone language model generated the answer "a fusion reaction", which is unrelated to the university context and demonstrates the limitations of relying solely on pretrained model knowledge.

The RAG system successfully retrieved information from the COS802 Study Guide and generated the correct answer:

"MIT 801 [Intro to Machine Learning]"

This example demonstrates how retrieval provides institution-specific knowledge that is not contained within the language model itself and highlights the value of grounding responses in authoritative university documents.

## Conclusion
The RAG system produced a substantially more accurate and contextually relevant answer than the standalone language model.

By retrieving information directly from university documents before answer generation, the system was able to provide institution-specific information that was unavailable to the baseline model.

This initial experiment provides evidence that Retrieval-Augmented Generation can improve the accuracy and relevance of answers generated from university documentation.

In [20]:
evaluation_questions = [

    "What is the prerequisite for COS802?",

    "What is the minimum entry requirement for admission to a master's degree?",

    "What is a supplementary examination?",

    "What mark is required to pass a supplementary examination?",

    "Who may write a special examination?",

    "What is a professional master's degree?",

    "Can admission to an honours programme be refused?",

    "What qualification is required for admission to an honours degree?",

    "What support services are available to students?",

    "What are the deliverables for the COS802 project?"
]

In [21]:
comparison_results = []

In [22]:
for question in evaluation_questions:

    print(f"\nProcessing: {question}")

    baseline_answer = generate_baseline_answer(
        question
    )

    rag_answer = generate_rag_answer(
        question
    )

    comparison_results.append(
        {
            "Question": question,
            "Baseline Answer": baseline_answer,
            "RAG Answer": rag_answer
        }
    )

print("Comparison complete.")


Processing: What is the prerequisite for COS802?

Processing: What is the minimum entry requirement for admission to a master's degree?

Processing: What is a supplementary examination?

Processing: What mark is required to pass a supplementary examination?

Processing: Who may write a special examination?

Processing: What is a professional master's degree?

Processing: Can admission to an honours programme be refused?

Processing: What qualification is required for admission to an honours degree?

Processing: What support services are available to students?

Processing: What are the deliverables for the COS802 project?
Comparison complete.


In [23]:
comparison_df = pd.DataFrame(
    comparison_results
)

comparison_df

,Question,Baseline Answer,RAG Answer
0,What is the prerequisite for COS802?,a fusion reaction,MIT 801 [Intro to Machine Learning]
1,What is the minimum entry requirement for admi...,a bachelor's degree in a related field,A bachelor's degree
2,What is a supplementary examination?,a secondary examination,a student is admitted to a supplementary exami...
3,What mark is required to pass a supplementary ...,0,50%
4,Who may write a special examination?,a teacher,students
5,What is a professional master's degree?,a degree in education,successful completion of a coursework programm...
6,Can admission to an honours programme be refused?,no,yes
7,What qualification is required for admission t...,a masters degree in english,A bachelor's degree
8,What support services are available to students?,adolescent mental health services,"User support (voice, walk-in, email, mail, web)"
9,What are the deliverables for the COS802 project?,a satellite,"project proposal, a final report, and a presen..."


In [24]:
for _, row in comparison_df.iterrows():

    print("\n" + "=" * 100)

    print("QUESTION:")
    print(row["Question"])

    print("\nBASELINE:")
    print(row["Baseline Answer"])

    print("\nRAG:")
    print(row["RAG Answer"])


QUESTION:
What is the prerequisite for COS802?

BASELINE:
a fusion reaction

RAG:
MIT 801 [Intro to Machine Learning]

QUESTION:
What is the minimum entry requirement for admission to a master's degree?

BASELINE:
a bachelor's degree in a related field

RAG:
A bachelor's degree

QUESTION:
What is a supplementary examination?

BASELINE:
a secondary examination

RAG:
a student is admitted to a supplementary examination in a module, in cases where: (a) a final mark of between 40% and 49% has been obtained; or (b) a final mark of 50% has been obtained, but the required subminimum in the examination has not been obtained

QUESTION:
What mark is required to pass a supplementary examination?

BASELINE:
0

RAG:
50%

QUESTION:
Who may write a special examination?

BASELINE:
a teacher

RAG:
students

QUESTION:
What is a professional master's degree?

BASELINE:
a degree in education

RAG:
successful completion of a coursework programme requiring a high level of theoretical engagement and intelle

In [25]:
comparison_df["Better Approach"] = ""

comparison_df

,Question,Baseline Answer,RAG Answer,Better Approach
0,What is the prerequisite for COS802?,a fusion reaction,MIT 801 [Intro to Machine Learning],
1,What is the minimum entry requirement for admi...,a bachelor's degree in a related field,A bachelor's degree,
2,What is a supplementary examination?,a secondary examination,a student is admitted to a supplementary exami...,
3,What mark is required to pass a supplementary ...,0,50%,
4,Who may write a special examination?,a teacher,students,
5,What is a professional master's degree?,a degree in education,successful completion of a coursework programm...,
6,Can admission to an honours programme be refused?,no,yes,
7,What qualification is required for admission t...,a masters degree in english,A bachelor's degree,
8,What support services are available to students?,adolescent mental health services,"User support (voice, walk-in, email, mail, web)",
9,What are the deliverables for the COS802 project?,a satellite,"project proposal, a final report, and a presen...",


In [26]:
comparison_df["Better Approach"] = [
    "RAG",
    "Tie",
    "RAG",
    "RAG",
    "RAG",
    "RAG",
    "RAG",
    "RAG",
    "RAG",
    "RAG"
]

comparison_df

,Question,Baseline Answer,RAG Answer,Better Approach
0,What is the prerequisite for COS802?,a fusion reaction,MIT 801 [Intro to Machine Learning],RAG
1,What is the minimum entry requirement for admi...,a bachelor's degree in a related field,A bachelor's degree,Tie
2,What is a supplementary examination?,a secondary examination,a student is admitted to a supplementary exami...,RAG
3,What mark is required to pass a supplementary ...,0,50%,RAG
4,Who may write a special examination?,a teacher,students,RAG
5,What is a professional master's degree?,a degree in education,successful completion of a coursework programm...,RAG
6,Can admission to an honours programme be refused?,no,yes,RAG
7,What qualification is required for admission t...,a masters degree in english,A bachelor's degree,RAG
8,What support services are available to students?,adolescent mental health services,"User support (voice, walk-in, email, mail, web)",RAG
9,What are the deliverables for the COS802 project?,a satellite,"project proposal, a final report, and a presen...",RAG


In [27]:
rag_wins = (
    comparison_df["Better Approach"] == "RAG"
).sum()

ties = (
    comparison_df["Better Approach"] == "Tie"
).sum()

baseline_wins = (
    comparison_df["Better Approach"] == "Baseline"
).sum()

total = len(comparison_df)

print(f"RAG Wins: {rag_wins}")
print(f"Ties: {ties}")
print(f"Baseline Wins: {baseline_wins}")

print(
    f"RAG Improvement Rate: {(rag_wins/total)*100:.1f}%"
)

RAG Wins: 9
Ties: 1
Baseline Wins: 0
RAG Improvement Rate: 90.0%


In [31]:
expected_keywords = {

    "What is the prerequisite for COS802?":
        ["MIT 801", "Machine Learning"],

    "What is the minimum entry requirement for admission to a master's degree?":
        ["honours degree", "NQF Level 8"],

    "What is a supplementary examination?":
        ["supplementary examination"],

    "What mark is required to pass a supplementary examination?":
        ["50%"],

    "Who may write a special examination?":
        ["student"],

    "What is a professional master's degree?":
        ["professional", "master's"],

    "Can admission to an honours programme be refused?":
        ["refused", "admission"],

    "What qualification is required for admission to an honours degree?":
        ["bachelor", "degree"],

    "What support services are available to students?":
        ["student", "support"],

    "What are the deliverables for the COS802 project?":
        ["proposal", "report", "presentation"]
}

In [32]:
scored_results = []

In [33]:
for _, row in comparison_df.iterrows():

    question = row["Question"]

    baseline_answer = row["Baseline Answer"]

    rag_answer = row["RAG Answer"]

    keywords = expected_keywords[question]

    baseline_score = keyword_score(
        baseline_answer,
        keywords
    )

    rag_score = keyword_score(
        rag_answer,
        keywords
    )

    scored_results.append({
        "Question": question,
        "Baseline Score": baseline_score,
        "RAG Score": rag_score
    })

In [34]:
scoring_df = pd.DataFrame(
    scored_results
)

scoring_df

,Question,Baseline Score,RAG Score
0,What is the prerequisite for COS802?,0.0,1.0
1,What is the minimum entry requirement for admi...,0.0,0.0
2,What is a supplementary examination?,0.0,1.0
3,What mark is required to pass a supplementary ...,0.0,1.0
4,Who may write a special examination?,0.0,1.0
5,What is a professional master's degree?,0.0,0.5
6,Can admission to an honours programme be refused?,0.0,0.0
7,What qualification is required for admission t...,0.5,1.0
8,What support services are available to students?,0.0,0.5
9,What are the deliverables for the COS802 project?,0.0,1.0


In [35]:
print(
    f"Average Baseline Score: "
    f"{scoring_df['Baseline Score'].mean():.2f}"
)

print(
    f"Average RAG Score: "
    f"{scoring_df['RAG Score'].mean():.2f}"
)

Average Baseline Score: 0.05
Average RAG Score: 0.70


In [36]:
rag_wins = (
    scoring_df["RAG Score"] >
    scoring_df["Baseline Score"]
).sum()

baseline_wins = (
    scoring_df["Baseline Score"] >
    scoring_df["RAG Score"]
).sum()

ties = (
    scoring_df["Baseline Score"] ==
    scoring_df["RAG Score"]
).sum()

print(f"RAG Wins: {rag_wins}")
print(f"Baseline Wins: {baseline_wins}")
print(f"Ties: {ties}")

RAG Wins: 8
Baseline Wins: 0
Ties: 2


In [38]:
presentation_df = comparison_df[
    [
        "Question",
        "Baseline Answer",
        "RAG Answer"
    ]
].copy()

presentation_df["Expected Keywords"] = presentation_df[
    "Question"
].map(
    lambda q: ", ".join(expected_keywords[q])
)

presentation_df["Winner"] = [
    "RAG",   # COS802 prerequisite
    "Tie",   # Master's admission
    "RAG",   # Supplementary examination
    "RAG",   # Pass mark
    "RAG",   # Special examination
    "RAG",   # Professional master's
    "RAG",   # Honours refusal
    "RAG",   # Honours qualification
    "RAG",   # Support services
    "RAG"    # COS802 deliverables
]

presentation_df = presentation_df[
    [
        "Question",
        "Expected Keywords",
        "Baseline Answer",
        "RAG Answer",
        "Winner"
    ]
]

presentation_df

,Question,Expected Keywords,Baseline Answer,RAG Answer,Winner
0,What is the prerequisite for COS802?,"MIT 801, Machine Learning",a fusion reaction,MIT 801 [Intro to Machine Learning],RAG
1,What is the minimum entry requirement for admi...,"honours degree, NQF Level 8",a bachelor's degree in a related field,A bachelor's degree,Tie
2,What is a supplementary examination?,supplementary examination,a secondary examination,a student is admitted to a supplementary exami...,RAG
3,What mark is required to pass a supplementary ...,50%,0,50%,RAG
4,Who may write a special examination?,student,a teacher,students,RAG
5,What is a professional master's degree?,"professional, master's",a degree in education,successful completion of a coursework programm...,RAG
6,Can admission to an honours programme be refused?,"refused, admission",no,yes,RAG
7,What qualification is required for admission t...,"bachelor, degree",a masters degree in english,A bachelor's degree,RAG
8,What support services are available to students?,"student, support",adolescent mental health services,"User support (voice, walk-in, email, mail, web)",RAG
9,What are the deliverables for the COS802 project?,"proposal, report, presentation",a satellite,"project proposal, a final report, and a presen...",RAG


In [39]:
summary_metrics = pd.DataFrame({
    "Metric": [
        "Documents Indexed",
        "Chunks Generated",
        "Top-5 Retrieval Accuracy",
        "RAG Wins",
        "Baseline Wins",
        "Ties"
    ],
    "Value": [
        9,
        1062,
        "90%",
        9,
        0,
        1
    ]
})

summary_metrics

,Metric,Value
0,Documents Indexed,9
1,Chunks Generated,1062
2,Top-5 Retrieval Accuracy,90%
3,RAG Wins,9
4,Baseline Wins,0
5,Ties,1


## Observations
The RAG system was compared against a standalone language model using ten university-related questions.

The standalone model frequently generated generic, inaccurate or irrelevant responses when asked institution-specific questions. Examples included identifying the prerequisite for COS802 as "a fusion reaction" and describing COS802 project deliverables as "a satellite".

By contrast, the RAG system consistently incorporated information retrieved from the university document corpus and produced contextually relevant answers. The RAG approach outperformed the standalone model on nine of the ten evaluation questions.

The results suggest that retrieval provides access to institution-specific knowledge that is unavailable to a standalone language model. This significantly improves answer relevance and grounding.

The Baseline vs RAG experiment demonstrated a clear advantage for Retrieval-Augmented Generation.

Across ten evaluation questions, the RAG system produced more accurate and institution-specific answers than the standalone language model. The retrieval component enabled access to authoritative university documents, reducing the reliance on pretrained model knowledge alone.

These findings support the central research hypothesis that Retrieval-Augmented Generation improves answer quality when responding to questions based on university documentation.

## Conclusion
The Baseline vs RAG evaluation demonstrated a clear advantage for Retrieval-Augmented Generation.

The standalone language model struggled to answer institution-specific questions that required knowledge from university regulations, policies and module documentation. The RAG system improved answer quality by grounding responses in retrieved evidence from the university document corpus.

The system achieved:

Top-5 Retrieval Accuracy: 90%
Average Baseline Score: 0.05
Average RAG Score: 0.70
RAG Wins: 8
Ties: 2
These findings support the research hypothesis that Retrieval-Augmented Generation improves answer quality, relevance and grounding when responding to questions based on university documentation.